# Practical 7: Data Wrangling for Aggregated Analysis
**Course:** BE05000231 - PDS (Semester V, GTU GSET)
**Course Outcome:** CO4

### Objective:
Wrangle security logs through IP aggregation, time-series resampling, multi-dimensional pivot tables, noise filtering (internal RFC 1918 IPs and verified search bots), and GeoIP threat intelligence enrichment.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')
import pandas as pd

from pipeline.parser import parse_file
from pipeline.cleaner import clean
from pipeline.labeler import load_rules, label_requests
from pipeline.wrangle import filter_noise, ip_summary, resample_traffic, create_pivots, enrich_geoip

LOG_PATH = Path('../tests/fixtures/sample_access.log')
RULES_PATH = Path('../rules/web_rules.yaml')
ALLOWLIST_PATH = Path('../rules/allowlist.yaml')


## 2. Ingest, Clean, and Label Telemetry

In [ ]:
raw_df, _ = parse_file(LOG_PATH)
cleaned_df = clean(raw_df)
rules = load_rules(RULES_PATH)
df = label_requests(cleaned_df, rules)
print(f'Total Ingested Records: {len(df)}')


## 3. Perimeter Noise Filtering

In [ ]:
filtered_df = filter_noise(df, allowlist_path=ALLOWLIST_PATH)
print(f'Retained Perimeter Events: {len(filtered_df)} (excluded internal RFC 1918 IPs and search spiders)')


## 4. Per-IP Aggregation Summary

In [ ]:
summary = ip_summary(filtered_df)
print('Top Attacking IPs:')
summary.head(8)


## 5. Time-Series Resampling (Traffic Rhythm)

In [ ]:
resampled = resample_traffic(filtered_df, freq='1min')
print('Resampled 1-Minute Traffic:')
resampled.head(10)


## 6. Security Cross-Tabulation Pivot Tables

In [ ]:
pivots = create_pivots(filtered_df)
print('HTTP Method by Attack Label:')
print(pivots.get('method_by_label'))

print('\nHTTP Status Code by Attack Label:')
print(pivots.get('status_by_label'))


## 7. GeoIP Threat Intelligence Enrichment

In [ ]:
enriched_df = enrich_geoip(filtered_df)
print('Enriched Geographic Attribution:')
enriched_df[['ip', 'geo_country', 'geo_iso', 'label']].drop_duplicates(subset=['ip']).head(8)


### Conclusion:
Practical 7 demonstrated aggregations, noise filtering, and geo-enrichment, transforming raw line-by-line log data into actionable security intelligence.